# D12 · Pipelines reproducibles — Solución

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea B · *Data Scientist* · Semana 13 · Soluciones de referencia.


## El problema que vamos a resolver

Imagina repetir a mano, cada vez: escalar el entrenamiento, escalar la prueba con los mismos
valores, entrenar, predecir… En cuanto el proceso crece, **algo se rompe o se hace mal**. El error
más peligroso es la **fuga de información** (*data leakage*) —el mismo enemigo de D8 y D11—: si
escalas usando **todos** los datos (entrenamiento + prueba) antes de dividir, el modelo "espía"
información de la prueba y tus resultados salen falsamente buenos.

Un **pipeline** resuelve esto: empaqueta los pasos en **un solo objeto** que se entrena y predice
de una vez, y que **siempre** aplica el escalado en el orden correcto, sin filtraciones.

Ejecuta la celda de preparación.


In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os
import urllib.request
import pandas as pd
from sklearn.model_selection import train_test_split

# Si el archivo no existe localmente (ej: en Colab), intentamos descargarlo desde GitHub
if not os.path.exists("compras_ml.csv"):
    try:
        url = "https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/B5-pipelines-reproducibles/compras_ml.csv"  # Reemplazar por la URL real del repositorio al publicar
        urllib.request.urlretrieve(url, "compras_ml.csv")
        print("compras_ml.csv descargado automáticamente.")
    except Exception:
        print("Aviso: No se pudo descargar compras_ml.csv automáticamente. Si estás en Colab, asegúrate de subirlo manualmente.")

df = pd.read_csv("compras_ml.csv")
X = df[["cantidad", "tamano_num"]]
y = df["monto_total"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
print(f"{len(df)} compras reales cargadas y divididas.")


Antes de los pipelines, veamos por qué el escalado importa. Probemos **K-vecinos (KNN)**, un
modelo que predice promediando las compras más **parecidas** (más cercanas). Como mide
**distancias**, si una variable tiene números mucho más grandes (el monto o la cantidad en comparación al
tamaño numérico del proveedor de 1 a 4), domina y el modelo empeora. Por tanto, es crítico aplicar un
escalado antes del KNN. Los pipelines nos aseguran que nunca olvidemos este paso.


In [ ]:
# (ilustracion) Probemos un modelo NUEVO: K-vecinos (KNN).
# Predice el monto de una compra promediando la de sus compras "vecinas" mas parecidas.
# Como mide DISTANCIAS, las escalas importan: cantidad (1-500) y tamano_num (1-4) no son directamente comparables.
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error

knn_crudo = KNeighborsRegressor(n_neighbors=3).fit(X_train, y_train)
mae_crudo = mean_absolute_error(y_test, knn_crudo.predict(X_test))
print(f"MAE de KNN sin escalar: {mae_crudo:.2f} CLP")


## 1. El pipeline: encadenar pasos en un solo objeto

Un **pipeline** une, en orden, varios pasos —típicamente un preprocesamiento y un modelo— en **una
sola pieza**:

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

pipe = Pipeline([
    ("escalador", StandardScaler()),
    ("modelo",    KNeighborsRegressor(n_neighbors=3)),
])
```

> 🧠 **Analogía.** Es como una **receta plastificada**: en vez de recordar cada paso suelto cada
> vez, tienes la secuencia fija. La sigues igual hoy, mañana y con datos nuevos, sin saltarte
> nada ni cambiar el orden.

A partir de aquí, el pipeline **se comporta como un modelo cualquiera**: tiene `fit` y `predict`.
Cuando llamas a `fit`, escala con el entrenamiento y entrena el modelo; cuando llamas a `predict`,
aplica el **mismo** escalado y predice. Esa disciplina automática es lo que evita las filtraciones.

> ⚠️ **Errores típicos**
> - **Invertir el orden de los pasos** (modelar antes de escalar).
> - **Escalar a mano por fuera** del pipeline "por si acaso": duplicas trabajo y reintroduces el
>   riesgo que el pipeline justamente elimina.


### ✍️ Ejercicio 1 — Construir el pipeline
Crea `pipe` como un `Pipeline` con dos pasos, en este orden: `("escalador", StandardScaler())` y
`("modelo", KNeighborsRegressor(n_neighbors=3))`. Completa la celda.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

pipe = Pipeline([
    ("escalador", StandardScaler()),
    ("modelo", KNeighborsRegressor(n_neighbors=3))
])

print("Pipeline creado exitosamente.")


In [ ]:
# ── Celda de chequeo · Ejercicio 1 ───────────────────────────────────────────
from sklearn.pipeline import Pipeline
try:
    assert isinstance(pipe, Pipeline), "pipe debe ser un objeto Pipeline."
    nombres = [n for n, _ in pipe.steps]
    assert nombres == ["escalador", "modelo"], \
        f"El pipeline debe tener dos pasos: 'escalador' y luego 'modelo'. Tienes: {nombres}"
    from sklearn.preprocessing import StandardScaler
    assert isinstance(pipe.steps[0][1], StandardScaler), "El primer paso debe ser un StandardScaler."
    print("✅ Correcto: creaste el pipeline en el orden adecuado (escalador -> modelo).")
except Exception as e:
    print("❌ Aun no. Verifica el orden y nombre de los pasos de tu pipeline.")
    print("   Pista:", e)


### ✍️ Ejercicio 2 — Entrenar y evaluar el pipeline
Un pipeline se usa **igual** que un modelo. Entrena `pipe` con `X_train`, `y_train` (`.fit`),
predice sobre `X_test` y calcula el MAE en prueba, guardándolo en `mae_pipe`. Compara con el KNN
sin escalar de arriba. Completa la celda.


In [ ]:
from sklearn.metrics import mean_absolute_error

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
mae_pipe = mean_absolute_error(y_test, y_pred)

print(f"MAE del pipeline (escalado + KNN) en prueba: {mae_pipe:.2f} CLP")


In [ ]:
# ── Celda de chequeo · Ejercicio 2 ───────────────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error
try:
    ref = Pipeline([("escalador", StandardScaler()),
                    ("modelo", KNeighborsRegressor(n_neighbors=3))]).fit(X_train, y_train)
    mae_ref = mean_absolute_error(y_test, ref.predict(X_test))
    assert abs(float(mae_pipe) - mae_ref) < 0.05, f"Tu MAE deberia ser ~{mae_ref:.2f} CLP."
    print(f"✅ Correcto. El MAE con escalado cayo a ~{mae_ref:.2f} CLP.")
except Exception as e:
    print("❌ Aun no. Asegurate de entrenar el pipeline con datos de train y evaluar sobre test.")
    print("   Pista:", e)


## 2. Validación cruzada: una evaluación más honesta

Evaluar con **una sola** división entrenamiento/prueba tiene un riesgo: el resultado depende de qué
compras cayeron, por azar, en la prueba. Con pocos datos o datos muy variables, eso puede fluctuar.

La **validación cruzada** (*cross-validation*) lo arregla: divide los datos en *k* partes (ej: 5),
entrena el modelo 5 veces usando cada vez una parte distinta como prueba y el resto como entrenamiento,
y luego promedia el error. Esto nos da una estimación mucho más honesta y robusta de cómo funcionará
el modelo con datos del futuro.


### ✍️ Ejercicio 3 — Evaluar con validación cruzada
Con `cv = KFold(n_splits=5, shuffle=True, random_state=42)`, calcula
`cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring="neg_mean_absolute_error")` y el MAE
promedio `mae_cv = -cv_scores.mean()`. Completa la celda.


In [ ]:
from sklearn.model_selection import cross_val_score, KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring="neg_mean_absolute_error")
mae_cv = -cv_scores.mean()

print(f"MAE promedio (validacion cruzada): {mae_cv:.2f} CLP")


In [ ]:
# ── Celda de chequeo · Ejercicio 3 ───────────────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score, KFold
try:
    assert len(cv_scores) == 5, f"La validacion cruzada de 5 cortes debe dar 5 puntajes; tienes {len(cv_scores)}."
    ref = Pipeline([("escalador", StandardScaler()),
                    ("modelo", KNeighborsRegressor(n_neighbors=3))])
    ref_cv = KFold(n_splits=5, shuffle=True, random_state=42)
    ref_scores = cross_val_score(ref, X, y, cv=ref_cv, scoring="neg_mean_absolute_error")
    ref_mae_cv = -ref_scores.mean()
    assert abs(float(mae_cv) - ref_mae_cv) < 0.05, f"Tu MAE de cv deberia ser ~{ref_mae_cv:.2f} CLP."
    print(f"✅ Correcto. El MAE estimado por validacion cruzada es ~{ref_mae_cv:.2f} CLP.")
    print("   Este error es mas realista porque evalua el modelo a lo largo de todo el dataset.")
except Exception as e:
    print("❌ Aun no. Revisa tu configuracion de cross_val_score y el promedio.")
    print("   Pista:", e)


## 3. Guardar el modelo para reutilizarlo

Entrenar un modelo cada vez que lo necesitas es absurdo (y poco reproducible). Lo correcto es
**entrenarlo una vez y guardarlo** en un archivo; después se **recarga** y se usa al instante. Se
hace con `joblib`, y lo bueno es que guardamos el **pipeline completo** (el escalador incluido).
Así, cuando lleguen datos nuevos, no tenemos que recordar escalarlos a mano: el pipeline lo hace solo.


### ✍️ Ejercicio 4 — Guardar, recargar y reutilizar
Guarda `pipe` con `joblib.dump(pipe, "modelo_compras.joblib")`, recárgalo en `pipe_cargado` con
`joblib.load(...)`, y úsalo para predecir una compra nueva con **cantidad 100 y tamano_num 2**,
guardando el resultado en `temp_nueva` (conservamos este nombre de variable para no romper chequeos legacy). Completa la celda.


In [ ]:
import joblib
import pandas as pd

# Guardar el pipeline en un archivo
joblib.dump(pipe, "modelo_compras.joblib")

# Recargar el pipeline
pipe_cargado = joblib.load("modelo_compras.joblib")

# Predecir una compra nueva (cantidad=100, tamano_num=2)
compra_nueva = pd.DataFrame({"cantidad": [100], "tamano_num": [2]})
temp_nueva = pipe_cargado.predict(compra_nueva)[0]

print(f"Prediccion de la compra recargada: {temp_nueva:.2f} CLP")


In [ ]:
# ── Celda de chequeo · Ejercicio 4 ───────────────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
import pandas as pd
try:
    ref = Pipeline([("escalador", StandardScaler()),
                    ("modelo", KNeighborsRegressor(n_neighbors=3))]).fit(X_train, y_train)
    esperado = ref.predict(pd.DataFrame({"cantidad": [100], "tamano_num": [2]}))[0]
    assert abs(float(temp_nueva) - esperado) < 0.1, \
        f"Para cantidad=100 y tamano_num=2 deberias estimar ~{esperado:.2f} CLP."
    import os
    assert os.path.exists("modelo_compras.joblib"), "Falta guardar el archivo 'modelo_compras.joblib'."
    print(f"✅ Correcto: guardaste, recargaste y usaste el pipeline. Prediccion: {temp_nueva:.2f} CLP. 💾")
except Exception as e:
    print("❌ Aun no. Revisa el nombre del archivo guardado y tu prediccion de la compra nueva.")
    print("   Pista:", e)


## 4. Cierre

Hoy convertiste tu trabajo en algo **ordenado, honesto y reutilizable**:

1. **Pipeline:** encadenaste escalado + modelo en un solo objeto, que se usa con `fit` / `predict`
   y aplica los pasos siempre en el orden correcto.
2. **Sin fugas:** entendiste que el pipeline escala usando solo la media/desviacion del conjunto
   de entrenamiento, protegiendo al conjunto de prueba de contaminarse (*data leakage*).
3. **Validación cruzada:** estimaste el error de manera mucho más robusta dividiendo y promediando.
4. **Persistencia:** exportaste tu pipeline completo a un archivo `.joblib` listo para desplegar.

En **D13 · Despliegue de modelos** daremos el cierre conceptual a esta ruta. Veremos cómo tomar
este archivo `.joblib` y exponerlo en una API web para que cualquier sistema de compras públicas
del Estado pueda hacer predicciones en tiempo real.

### Mini-glosario
- **Pipeline:** cadena de transformadores + estimador tratado como un único objeto.
- **Fuga de información (*data leakage*):** contaminar el entrenamiento con info del test.
- **Validación cruzada:** evaluar en k subdivisiones rotativas para robustecer la métrica.
- **Persistencia (`joblib`):** guardar objetos de Python en disco.

---
*Fuente de datos: Dirección de Compras y Contratación Pública (ChileCompra / MercadoPúblico).*
*Contenido bajo licencia CC BY 4.0 · Formación Pública.*
